# Gold — Economic Dashboard

Joins Silver tables into a single monthly snapshot for the Power BI dashboard.

**Sources:** `silver.exchange_rates`, `silver.central_bank_indicators`, `silver.gdp_indicators`  
**Output:** `gold.economic_dashboard` (Delta table)  
**Grain:** One row per calendar month  

| Column | Description |
|---|---|
| `date` | First day of the month |
| `avg_iskusd` | Monthly average ISK/USD rate |
| `avg_iskeur` | Monthly average ISK/EUR rate |
| `avg_eurusd` | Monthly average EUR/USD rate |
| `avg_omx` | Monthly average OMX Iceland All-Share Index |
| `policy_rate` | End-of-month Central Bank policy rate (%) |
| `cpi` | CPI year-on-year inflation (%) |
| `gdp_yoy_growth` | Quarterly GDP YoY growth (%) mapped to each month |

In [ ]:
from pyspark.sql import functions as F

df_fx = spark.read.format("delta").table("silver.exchange_rates")
df_cb = spark.read.format("delta").table("silver.central_bank_indicators")
df_gdp = spark.read.format("delta").table("silver.gdp_indicators")

print(f"Exchange rates:  {df_fx.count()} rows")
print(f"Central bank:    {df_cb.count()} rows")
print(f"GDP:             {df_gdp.count()} rows")

In [ ]:
# Monthly averages for FX rates and OMX index
df_fx_monthly = df_fx.groupBy(
    F.year("date").alias("year"),
    F.month("date").alias("month"),
).agg(
    F.round(F.avg("close_iskusd_x"), 6).alias("avg_iskusd"),
    F.round(F.avg("close_iskeur"),   6).alias("avg_iskeur"),
    F.round(F.avg("close_eurusd_x"), 6).alias("avg_eurusd"),
    F.round(F.avg("close__omx"),     2).alias("avg_omx"),
)

# End-of-month policy rate
df_policy = (
    df_cb.filter(F.col("metric") == "policy_rate")
    .groupBy(F.year("date").alias("year"), F.month("date").alias("month"))
    .agg(F.round(F.last("value"), 4).alias("policy_rate"))
)

# Monthly CPI
df_cpi = (
    df_cb.filter(F.col("metric") == "cpi")
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
    .select("year", "month", F.col("value").alias("cpi"))
)

# GDP quarter → month mapping
df_gdp_monthly = df_gdp.select("year", "q", F.col("value").alias("gdp_yoy_growth"))

# Join everything
df_gold = (
    df_fx_monthly
    .join(df_policy, ["year", "month"], "left")
    .join(df_cpi,    ["year", "month"], "left")
    .withColumn(
        "q",
        F.when(F.col("month").isin(1, 2, 3), 1)
         .when(F.col("month").isin(4, 5, 6), 2)
         .when(F.col("month").isin(7, 8, 9), 3)
         .otherwise(4)
    )
    .join(df_gdp_monthly, ["year", "q"], "left")
    .drop("q")
    .withColumn(
        "date",
        F.to_date(F.concat_ws("-", F.col("year"), F.col("month"), F.lit("1")), "yyyy-M-d")
    )
    .orderBy("year", "month")
)

df_gold.show(10)

In [ ]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.economic_dashboard")

print(f"Saved to gold.economic_dashboard — {df_gold.count()} rows")